# ESdE Adultos 2023: Test Exploratory analysis

~~This notebook is for exploratory work with the extracted INE microdata. Reusable extraction, loading, and codebook functions stay in `ine_health_data/`; calculations and visual inspection belong here.~~

Run the notebook from the repository root. 

The raw ESdE file and `references/metadata/esde_adulto_2023.json` must already exist. Otherwise run `ine_health_data.pipeline.start_sequence()`

In [ ]:
import pandas as pd
from ine_health_data.pipeline import add_value_labels, load_variables

## General population analysis
(population represented)

Stablishing the population that the dataset represents before realizing any in depth analysis.

Main target variables should be age, sex, and location.

In [2]:
analysis_variable = ["EDADa","SEXOa","CCAA"] 
raw_data = load_variables(variables=analysis_variable)

### Data integrity

The variable `EDADa` is the only numeric variable loaded, which includes special `999` code for "Not answered" value.
For `SEXOa`,`CCAA`, There are no possible "Not answered" special values.

In [38]:
rows_count          = len(raw_data)
nonfull_rows_count  = raw_data.isna().any(axis=1).sum()
age_not_answered_c  = raw_data["EDADa"].eq(999).sum()

pd.Series({
    "Total rows": rows_count,
    "Rows any with empty": nonfull_rows_count,
    "Not answered age rows": age_not_answered_c
}).to_frame(name="n")

,n
Total rows,21032
Rows any with empty,0
Not answered age rows,0


The dataset for `SEXOa`, `CCAA` and `EDADa` does **not** contain any empty or "Not answered" value.

This implication is carried into following python cells, avoiding the need for unnecessary filtering.

### Age and Sex analysis

#### General Age

In [57]:
non_ccaa_df = raw_data[["EDADa","SEXOa"]]

age_summary = (
    non_ccaa_df.loc[non_ccaa_df["EDADa"].ne(999), "EDADa"]
    .describe(percentiles=[0.25,0.5,0.75])
    .to_frame().T
    .rename(index={"EDADa": "General age data"})
    .round(2)
)
display(age_summary)


,count,mean,std,min,25%,50%,75%,max
General age data,21032.0,54.49,19.14,15.0,40.0,55.0,69.0,103.0


#### Age by Sex

In [59]:

labeled_national_df = add_value_labels(non_ccaa_df)

age_by_sex_freq = labeled_national_df["SEXOa_label"].value_counts(normalize=True).round(4)
age_by_sex_desc = (
    labeled_national_df.groupby("SEXOa_label")["EDADa"]
    .describe(percentiles=[0.25,0.5,0.75])
    .round(2)
)
age_by_sex_summary = (
    pd.concat([age_by_sex_freq, age_by_sex_desc], axis=1)
    .rename(columns={"SEXOa_label": "sex"}).rename_axis("age by sex")

)
display(age_by_sex_summary)

,proportion,count,mean,std,min,25%,50%,75%,max
age by sex,,,,,,,,,
Mujer,0.5397,11352.0,55.92,19.49,15.0,41.0,56.0,71.0,103.0
Hombre,0.4603,9680.0,52.8,18.59,15.0,39.0,53.0,67.0,98.0


#### Age per Location

In [ ]:
df_labeled = add_value_labels(raw_data)

age_by_ccaa = (
   df_labeled.groupby("CCAA_label")["EDADa"]
    .describe() # .drop(columns=["std","min","max"])
    .round(2)
    .reset_index()
    .sort_values("count", ascending=False).rename(columns={"CCAA_label":"CCAA"})
)
display(age_by_ccaa)

,CCAA_label,count,mean,std,min,25%,50%,75%,max
0,Andalucía,2674.0,52.67,19.12,15.0,39.0,53.0,67.0,99.0
10,Comunitat Valenciana,2059.0,54.55,19.03,15.0,40.0,55.0,70.0,99.0
13,"Madrid, Comunidad de",1876.0,53.82,19.6,15.0,39.0,53.0,69.0,103.0
8,Cataluña,1802.0,54.32,19.0,15.0,40.0,55.0,69.0,102.0
17,País Vasco,1364.0,55.8,19.54,15.0,42.0,56.0,71.0,98.0
7,Castilla y León,1344.0,57.05,19.14,15.0,44.0,58.0,71.0,99.0
11,Extremadura,1099.0,55.65,19.2,15.0,41.0,56.0,71.0,101.0
15,"Murcia, Región de",1080.0,52.34,18.81,15.0,38.0,53.0,67.0,94.0
1,Aragón,1044.0,55.73,19.67,15.0,41.0,56.0,71.0,99.0
6,Castilla - La Mancha,1030.0,55.31,19.57,15.0,41.0,55.0,70.75,95.0
